
# Determining the Z-Index with SMT Solvers

In this notebook, we will
look at SMT solvers and see how they can be used to formalize and solve problems. 
Particularly, we will show how we can determine the Z Index of a molecule. 
We will write a program to solve the matching problem to introduce you to Z3, a state-of-the-art SMT solver,
and its capabilities.


**Instructions:**
1. To get started, click on File on the top left and click "Save a copy in Drive."
This will give you an editable version of this document that you can use.
2. If you press `CMD`+`Enter` it runs the cell, and if you press `Shift`+`Enter` it runs the cell and goes to the next one.
3. Make sure you run all cells as you go through the notebook; some cells will not work properly unless the previous one
has been run too.
4. If you disconnect or are inactive for some time you should run all of the cells again.

## 0. Preliminaries (you should run this cell but there is no need to read it)

In [ ]:
!pip install z3-solver
!pip install git+https://github.com/crrivero/FormalMethodsTasting.git#subdirectory=core
from z3 import *
from tofmcore import showSolver, list_all_solutions
from IPython.display import clear_output
clear_output()

## Encoding constraints in Z3

The goal of this notebook is to teach you about formal methods;
particularly, how you can use existing formal verification tools
(in this case, Z3) to analyze and solve your own problems.
Before we get started, let's look at some basic things we can do with Z3.

We can also add arithmetic constraints to boolean variables:

$$ x_1 + x_2 + x_3 = 1 $$

To model these types of constraints, Z3 converts True/False boolean values to 1/0 integer values.

In [ ]:
s = Solver() # initialize solver

# initialize variables
x1 = Bool('x_1')
x2 = Bool('x_2')
x3 = Bool('x_3')

s.add( x1 + x2 + x3 == 1 )

print( s )

Note how Z3 uses If conditions to convert boolean variables into integers.
Our custom printer function prints these without the if conditions for better readability.

In [ ]:
showSolver( s )

In [ ]:
print( s.check() ) # check if satisfiable
print( s.model() ) # print solution

Now it's your turn! Complete the code in the cell below to find a solution satisfying the following constraints:

$$ x_1 \lor x_2 \lor x_3 $$
$$ x_1 + x_2 = x_3 $$
$$ \neg x_1 \implies x_3 $$

In [ ]:
s = Solver() # initialize solver

# initialize variables
x1 = Bool('x_1')
x2 = Bool('x_2')
x3 = True # REPLACE THIS LINE

s.add( Or( x1, x2 ) ) # REPLACE THIS LINE
s.add( x1 + x2 == 1 ) # REPLACE THIS LINE
s.add( x2 == x2 )  # REPLACE THIS LINE

showSolver( s )
print( s.check() )
print( s.model() )

Note that Z3 does not support this boolean-to-integer conversion for every operator. For example, adding the constraint `x1 >= 1` or `x1 == 1` to our solver would give an error. Proper usage dictates using integers when using arithmetic constraints, but to keep things simple for our introductory notebook we will still work with booleans.

### Integers

Let's use Z3 to solve problems involving integers. Let's start with something simple: find $x$ such that

$$2x + 5 = 15$$

In [ ]:
# Initialize variables

x = Int('x') # declairing that x is an integer named 'x'

# Initialize Z3 solver
s = Solver()

s.add( 2*x + 5 == 15 ) # add the equation

print(s)
print(s.check())
print(s.model())

Now let's try to check whether that's the only solution. We can do this by adding the following constraint to the solver:

$$x \not= 5$$

If the solver returns "**unsat**" then $x=5$ is the only solution.
Try it yourself by completing the code in the cell below.

In [ ]:
s.add( x == 5 ) # REPLACE THIS LINE
s.check()

In [ ]:
### SHOULD BE IMPORTED ###
def draw_all_matchings(s, all_sols, num_center):
  # Draw in a grid with 3 columns
  rows = (len(all_sols) // 3) + 1
  fig, axs = plt.subplots(rows,3, figsize=(12,12))
  axes = axs.flatten()

  for i, sol in enumerate(all_sols):
    draw_single_matching(sol, axes[i], num_center)

def draw_single_matching(m, ax, num_center):
  # Create the edges and the matchings from the solution
  edges = []
  matching = []
  for i in m:
    val = m[i]
    a = str(i)[3]
    b = str(i)[4]
    edges.append((a, b))
    if val:
      matching.append((a, b))

  # Create the graph from the edges.
  G = nx.Graph()
  G.add_edges_from(sorted(edges)) # Sorting ensures consisten layout of nodes
  edge_colors = ['red' if e in matching or (e[1], e[0]) in matching else 'black' for e in G.edges()]

  # Draw graph aligned to grid
  draw_chemical_graph(G, edge_colors, [str(i+1) for i in range(num_center)], ax)

### Written by Gemini ###
def draw_chemical_graph(G, edge_colors, centerline_nodes, ax):
    """
    centerline_nodes: a list of nodes to be placed on the y=0 axis.
    """
    pos = {}

    # 1. Position the centerline nodes
    for i, node in enumerate(centerline_nodes):
        pos[node] = (i, 0)

    # 2. Position the children above/below
    for parent in centerline_nodes:
        # Find neighbors not already in the centerline
        children = [n for n in G.neighbors(parent) if n not in centerline_nodes]

        for i, child in enumerate(children):
            # Alternate: even index above (1), odd index below (-1)
            x_offset = pos[parent][0]
            y_offset = 1 if i % 2 == 0 else -1
            pos[child] = (x_offset, y_offset)

    # Draw the graph
    nx.draw(G, pos, with_labels=True, node_size=700,
            node_color='white', edge_color=edge_colors, ax=ax)

## Matching using Z3

In this notebook, we will look at an application of the matching problem in chemistry. 
We will see how the Z-index chemical graph corresponds to the boiling point of the molecule.

The molecule we will be looking at are three different isomers of Hexane, $ C_6H_{14} $:

IMAGE HERE


### The Z Index

The Z Index of a molecule is the number of matchings within its chemical graph. 
A chemical graph is structure where the nodes represent atoms, and edges represent the bonds. Hydrogen atoms are excluded.

Below is the chemical graph of the first isomer we will be looking at:

IMAGE HERE

A matching is a selection of the edges such each node has at most one of their edges selected. We will use Z3 to find all possible matchings for this isomer. 


#### Step 1: Make boolean values for each edge#### Step 1: Make boolean variables for each edge

In our solution, an edge with the value of True will represent an edge that has been selected in our matching.

In [ ]:
# In the graph above, carbon 1 shares an edge with carbon 2, so we will represent that with the variable e_12
e12 = Bool('e_{12}')

# We follow the same conventiion for all other edges in the graph
e23 = Bool('e_{23}')
e34 = Bool('e_{34}')
e25 = Bool('e_{25}')
e26 = Bool('e_{26}')

vars = [e12, e23, e34, e25, e26]

#### Step 2: Initialize Solver

In [ ]:
s = Solver()

#### Step 3: Write Constraints

In [ ]:
# Carbon atom 2 can have at most one edge adjacent to it in the matching
s.add(e12 + e23 + e25 + e26 <= 1)

# Carbon atom 3 can have at most one edge adjacent to it in the matching
s.add(e23 + e34 <= 1)

# The rest of the atoms have only one edge, so no constraints are necessary!

# Let's see what the constraints look like
showSolver(s)


#### Step 4: Check if a solution exists

In [ ]:
print( s.check() )

Great! Now we can determine how many matchings there are, and visualize them. We have defined a function to find and show all matchings

In [ ]:
all_solutions = list_all_solutions(s, vars)
print( 'Number of matchings:', len(all_solutions) )
draw_all_matchings(s, all_solutions, 4)

Great! We have successfully determined the Z Index of this molecule.

Now its your turn! **Complete the code below** to determine the Z Index of the next isomer:

IMAGE HERE

In [ ]:
# Initialize Solver
s = Solver()

# In the graph above, carbon 1 shares an edge with carbon 2, so we will represent that with the variable e_12
e12 = Bool('e_{12}')

# We follow the same conventiion for all other edges in the graph
e23 = Bool('e_{23}')
e34 = Bool('e_{34}')
e45 = Bool('e_{45}')
e26 = Bool('e_{26}')
vars = [e12, e23, e34, e45, e26]

# Carbon atom 2 can have at most one edge adjacent to it in the matching
s.add(False) # REPLACE THIS LINE

# Carbon atom 3 can have at most one edge adjacent to it in the matching
s.add(False) # REPLACE THIS LINE

# Do the same for Carbon 4
s.add(False) # REPLACE THIS LINE

# The rest of the atoms have only one edge

# Let's see what the constraints look like
showSolver(s)


In [ ]:
print( s.check() )

In [ ]:
all_solutions = list_all_solutions(s, vars)
print( 'Number of matchings:', len(all_solutions) )
draw_all_matchings(s, all_solutions, 5)

Lastly, let's determine the Z Index of one more isomer, and compare it to their boiling points

**Complete the code below** to determine the Z Index of the final isomer:

IMAGE HERE

In [ ]:
# Initialize Solver
s = Solver()

# In the graph above, carbon 1 shares an edge with carbon 2, so we will represent that with the variable e_12
e12 = Bool('e_{12}')

# We follow the same conventiion for all other edges in the graph
e23 = Bool('e_{23}')
e34 = Bool('e_{34}')
e45 = Bool('e_{45}')
e56 = Bool('e_{56}')
vars = [e12, e23, e34, e45, e56]

# Carbon atom 2 can have at most one edge adjacent to it in the matching
s.add(False) # REPLACE THIS LINE

# Carbon atom 3 can have at most one edge adjacent to it in the matching
s.add(False) # REPLACE THIS LINE

# Do the same for Carbon 4
s.add(False) # REPLACE THIS LINE

# Do the same for Carbon 5
s.add(False) # REPLACE THIS LINE

# The rest of the atoms have only one edge

# Let's see what the constraints look like
showSolver(s)

In [ ]:
print( s.check() )

In [ ]:
all_solutions = list_all_solutions(s, vars)
print( 'Number of matchings:', len(all_solutions) )
draw_all_matchings(s, all_solutions, 6)

Great! Now we can compare the results to show that the Z-Index correlates to boiling point.
| Isomer | Z Index | Boiling Point |
| ------ | ------- | ------------- |
| 1      | 9       | 49.7          |
| 2      | 11      | 60.2          |
| 3      | 13      | 68.7          |


### Great Work! You have written a program to solve the matching problem using Z3!


####If you'd like to continue your Z3 journey, you can start with this guide to learn more:
https://ericpony.github.io/z3py-tutorial/guide-examples.htm